In [1]:
from datetime import date, datetime

import pandas as pd
from sqlalchemy import select

In [2]:
from database import Database

db = Database()

In [ ]:
query = (
    select(db.reports_bookingsfinancialdata)
    .outerjoin(
        db.reports_hotelsfinancials,
        db.reports_hotelsfinancials.c.booking_id
        == db.reports_bookingsfinancialdata.c.booking_code,
    )
    .where(
        db.reports_bookingsfinancialdata.c.check_in.between(
            date(2024, 3, 1), date(2025, 2, 28)
        ),
        db.reports_hotelsfinancials.c.booking_id.is_(None),
    )
)

bookings = pd.read_sql(sql=query, con=db.engine.connect())

In [ ]:
bookings.shape

In [27]:
clients_credential = pd.read_sql(
    sql=select(db.clients_credential), con=db.engine.connect()
)

In [28]:
clients_credential["id"] = pd.to_numeric(clients_credential["id"], errors="coerce")
clients_credential = clients_credential.dropna(subset=["id"]).assign(
    id=lambda x: x["id"].astype(int)
)
clients_credential.to_csv("clients_credential.csv", index=False)

In [34]:
import time

start_time = time.time()

reports_hotelsfinancials = pd.read_sql(
    sql=select(db.reports_hotelsfinancials).limit(1000000), con=db.engine.connect()
)

end_time = time.time()

duration = end_time - start_time
print(f"Query execution time: {duration:.4f} seconds")

Query execution time: 39.3753 seconds


In [1]:
import pandas as pd

# Example DataFrame for rules
rules = pd.DataFrame({
    'rule_id': [1, 2, 3],
    'credential_list': [[1, 2, 4, 5], [1, 3, 4], [1, 2, 3, 6]]
})

# Example DataFrame for credential_group (Group 4 as an example)
credential_group = pd.DataFrame({
    'group_id': [4],
    'credential_list': [[1, 2, 3, 4]]  # This is the group you are matching against
})

# Convert the group credentials to a set (assuming only one group for this example)
group_credentials = set(credential_group['credential_list'][0])

# Function to check if the rule's credentials are a subset of the group
def is_subset_of_group(rule_credentials, group_credentials):
    return set(rule_credentials).issubset(group_credentials)

# Filter the rules where the credential list is a subset of the group credentials
filtered_rules = rules[rules['credential_list'].apply(lambda x: is_subset_of_group(x, group_credentials))]

print(filtered_rules)


   rule_id credential_list
1        2       [1, 3, 4]


In [ ]:
nautalia_group = 